In [1]:
import pandas as pd
import pdfplumber
import re

In [2]:
PDF_PATH = 'Suggested-Premium-Rates-2025-26-full-report.pdf'

pdf = pdfplumber.open(PDF_PATH)
print(f'Total pages: {len(pdf.pages)}')

Total pages: 92


In [3]:
# Extract rows from 'Summary of Suggested Rates' tables (pages 16-31).
# Each table row has 5 columns from pdfplumber:
#   [0] ANZSIC06 class code (4-digit)
#   [1] Description + relativity group (int) + estimated wages (float), sometimes
#       split across newlines for long descriptions
#   [2] Claim freq rel + attritional payment rel + 2025/26 & 2024/25 selected relativities
#   [3] 2025/26 suggested premium rate (%)
#   [4] 2024/25 suggested premium rate (%) + premium rate change (%)

rows = []
for pg_idx in range(15, 31):
    page = pdf.pages[pg_idx]
    tables = page.extract_tables()
    if not tables:
        continue
    for row in tables[0][1:]:  # skip header row
        code = (row[0] or '').strip()
        if not code or not code[0].isdigit():
            continue

        col1 = (row[1] or '').strip()
        # Find relativity group (int) and estimated wages (float) embedded in the text
        m = re.search(r'(\d+)\s+([\d,]+\.\d+)\s*', col1)
        if not m:
            print(f'WARN: cannot parse col1 for {code}: {repr(col1)}')
            continue

        rel_group = int(m.group(1))
        wages = float(m.group(2).replace(',', ''))

        # Description is everything around the matched numbers
        desc_before = col1[:m.start()].strip()
        desc_after = col1[m.end():].strip()
        description = f'{desc_before} {desc_after}'.strip() if desc_after else desc_before
        description = re.sub(r'\s+', ' ', description)

        rate_2526 = float((row[3] or '').replace('%', '').strip())

        col4_parts = (row[4] or '').replace('%', '').strip().split()
        rate_2425 = float(col4_parts[0])
        rate_change = float(col4_parts[1]) if len(col4_parts) >= 2 else None

        rows.append({
            'anzsic_code': code,
            'description': description,
            'relativity_group': rel_group,
            'estimated_wages_millions': wages,
            'suggested_premium_rate_pct': rate_2526,
            'prior_year_rate_pct': rate_2425,
            'rate_change_pct': rate_change,
        })

df = pd.DataFrame(rows)
print(f'{len(df)} rows parsed')
df.head()

506 rows parsed


,anzsic_code,description,relativity_group,estimated_wages_millions,suggested_premium_rate_pct,prior_year_rate_pct,rate_change_pct
0,0111,Nursery Production (Under Cover),1,5.37,2.16,2.17,-0.4
1,0112,Nursery Production (Outdoors),1,12.17,2.16,2.17,-0.4
2,0113,Turf Growing,1,0.06,2.16,2.17,-0.4
3,0114,Floriculture Production (Under Cover),1,0.74,2.16,2.17,-0.4
4,0115,Floriculture Production (Outdoors),1,9.49,2.16,2.17,-0.4


In [4]:
df[['suggested_premium_rate_pct', 'prior_year_rate_pct', 'rate_change_pct']].describe()

,suggested_premium_rate_pct,prior_year_rate_pct,rate_change_pct
count,506.000000,506.000000,506.000000
mean,2.173557,2.004348,9.354743
std,1.319329,1.227475,10.612003
min,0.400000,0.400000,-14.230000
25%,1.240000,1.040000,-0.400000
50%,2.160000,1.980000,5.840000
75%,2.880000,2.640000,19.330000
max,12.350000,10.000000,33.190000


In [5]:
# Spot-check first, last, and wrapped-description rows
spot_checks = ['0111', '1320', '1521', '9602', '9603']
df[df['anzsic_code'].isin(spot_checks)][['anzsic_code', 'description', 'relativity_group', 'suggested_premium_rate_pct']]

,anzsic_code,description,relativity_group,suggested_premium_rate_pct
0,0111,Nursery Production (Under Cover),1,2.16
93,1320,"Leather Tanning, Fur Dressing and Leather Prod...",17,2.88
110,1521,Corrugated Paperboard and Paperboard Container...,20,1.24
504,9602,Undifferentiated Goods- Producing Activities o...,114,1.13
505,9603,Undifferentiated Service- Producing Activities...,114,1.13


In [6]:
df.to_parquet('TAS_WIC.parquet', index=False)
print(f'Saved TAS_WIC.parquet ({len(df)} rows)')

Saved TAS_WIC.parquet (506 rows)
